<h1>MAIN DIMENSIONS</h1>

In [10]:
import math


D = 0.085                   # Diameter of the stator stack = 8.5 cm
B_delta_1 = 1.1             # Peak fundamental air gap flux density = 1 T
As1rms = 40                 # Linear current density 50 kA/m
f = 50                      # Frequency of the fed current [Hz]
p = 2                       # Number of pole pairs [-]
U_l_l = 50                  # Line to line RMS voltage [V]
eta = 0.876                 # Efficiency of the motor [-]
cos_delta_s = 0.92          # Power factor [-]
n_r = 1500                  # mechanical speed of the rotor [okr/min]
P2 = 1000                   # Output mechanical power [W]
aspect_ratio = 2            # Ratio between stack length and pole pitch [-]
Do_D_ratio = 1.6            # Ratio to obtain the outer stator diameter [-]
Di = 28*1e-3                # Rotor inner shaft diameter


eta_delta = ( eta / (0.25 + 0.75 * eta) )
print(f"Eta delta  eta_delta = {eta_delta:.2f} ")
l = P2 / ( (math.pi**2 / (math.sqrt(2) * p)) * B_delta_1 * As1rms * 1000 * f * (D**2) * eta_delta * cos_delta_s )
print(f"Stack length l = {l:.2f} m" )
delta = ( 0.2 + 0.03*math.sqrt((D*1e3 * l*1e3 ) / 2) ) * 1e-3   # This is meters now [m]
print(f"Air gap length  delta = {delta * 1e3:.2f} mm")
T_n = (P2 / (n_r * math.pi) ) * 30
print(f"Nominal Torque  Tn = {T_n:.2f} Nm")
tau_p = l / aspect_ratio
print(f"Pole pitch tau_p = {tau_p:.2f} m")
Do = Do_D_ratio * D
print(f"Outer Diameter of the stator Do = {Do:.2f} m")





Eta delta  eta_delta = 0.97 
Stack length l = 0.02 m
Air gap length  delta = 1.08 mm
Nominal Torque  Tn = 6.37 Nm
Pole pitch tau_p = 0.01 m
Outer Diameter of the stator Do = 0.14 m


<h1> DETERMINATION OF THE PERMANENT MAGNET OPERATING POINT </h1>

In [35]:
#To start the calculation a magnet material needs to be selected first. Choice for today is N40 Neodymium from Arnold Magnetic Technologies available at: https://www.arnoldmagnetics.com/products/neodymium-iron-boron-magnets/

#Extracted from the datasheet:
Br = 1250e-3                        # Remanence [T]
Hc = 951e3                          # Coercivity [A/M]
#Choosen as a free parameter:
B_delta_m = 0.85                    # Air gap flux density [T]
mi_0 = 4 * math.pi * 1e-7           # Permeability of free space [H/m]
k_c = 1.02                          # Carter factor [-] , see Chapter 13
k_sat = 1.2                         # Saturation factor [-], see Chapter 13
h_m__delta_marked_ratio = 3         # Ration of magnet thickness and delta_m [-]
alpha_m = 130                       # Electrical span of the magnet pole [°]
mi_r = Br / (Hc * mi_0)             # Relative permeability [H/m]
print(f"Relative permeability mi_r = {mi_r:.2f} H/m")
delta_marked = k_c * k_sat * delta  # Virtual representation of the air gap [m]
print(f"Delta marked delta_marked = {delta_marked * 1e3:.3f} mm")
B_delta_m_1 = (4/math.pi) * B_delta_m * math.sin((alpha_m / 2) * (math.pi / 180))                     # Fundamental component of the air gap flux density of the magnet [T]
print(f"Fundamental component of permanent magnet air gap flux density B_delta_m_1 = {B_delta_m_1:.2f} T")
h_m = (delta_marked * mi_r) / ((Br / B_delta_m) - 1)    #Thickness of the magnet [m]
print(f"Thickness of the permanent magnet  h_m = {h_m * 1e3:.2f} mm")
b_m = math.radians(alpha_m) * (D / (2*p))         # Width of the magnet [m]
print(f"Width of the permanent magnet  b_m = {b_m:.2f} m")



Relative permeability mi_r = 1.05 H/m
Delta marked delta_marked = 1.323 mm
Fundamental component of permanent magnet air gap flux density B_delta_m_1 = 0.98 T
Thickness of the permanent magnet  h_m = 2.94 mm
Width of the permanent magnet  b_m = 0.05 m


<h1> CHOICE OF THE SLOT/POLE COMBINATION</h1>

In [8]:
# SWAT-EM is used for a quick preview of possible combinations and based on some research online fractional slot windings seems to do the best in regards to cogging torque which will later be important when implementing control algorithms
from fractions import Fraction
Q_s = 18                                # Number of slots [-]
m = 3                                   # Number of phases [-]
q = Q_s / (2*p*m)                       # Number of slots per pole per phase [-]
frac_q = Fraction(q).limit_denominator()
a_q = frac_q.numerator
b_q = frac_q.denominator
y = 4                                   # Coil pitch [-]
tau_p_slots = Q_s / (2*p)
print(f"Number of slots per pole per phase q = {q:.2f} ")
t = 2                                   # Greatest common denominator GCD(Q_s, p) = GCD(18, 2) = 2
Q_t = Q_s / t                           # Number of voltage phasors [-]
print(f"Number of voltage phasors Q_t = {Q_t} ")
alpha = p * 360 / Q_s                   # Electrical degree between two consecutive slots [°]
print(f"Electrical degree between two consecutive slots is {alpha:.2f} °")
q_marked = Q_s / (m*t)                  # Apparent number of slots per pole per phase [-]
print(f"Apparent number of slots per pole per phase q_marked = {q_marked} ")
alpha_marked = 180 / (m * q_marked)     # Apparent electrical degree between consecutive slots [°]
print(f"Apparent electrical degree between two consecutive slots is = {alpha_marked} ")
f_d = math.sin(q_marked * (alpha_marked / 2) * (math.pi / 180) ) / (q_marked * math.sin((alpha_marked / 2) * (math.pi / 180) ))    #Distribution factor of the windings [-]
print(f"Distribution winding factor fd = {f_d:.2f} ")
f_p = math.sin((y / tau_p_slots) * (math.pi/2) )
print(f"Pitch winding factor = {f_p:.2f} ")
f_w = f_d * f_p
print(f"Winding factor f_w = {f_w:.3f} ")
C_t = (2*p*Q_s) / math.lcm(Q_s,2*p)     # Goodness factor (lower values indicate smaller values of cogging torque)
print(f"Goodness factor C_t = {C_t:.2f} ")
p_marked = 2*p / b_q
print(f"Number of radial force pole pairs p_marked = {p_marked} ")

Number of slots per pole per phase q = 1.50 
Number of voltage phasors Q_t = 9.0 
Electrical degree between two consecutive slots is 40.00 °
Apparent number of slots per pole per phase q_marked = 3.0 
Apparent electrical degree between two consecutive slots is = 20.0 
Distribution winding factor fd = 0.96 
Pitch winding factor = 0.98 
Winding factor f_w = 0.945 
Goodness factor C_t = 2.00 
Number of radial force pole pairs p_marked = 2.0 


<h1>SKEWING OF STATOR SLOTS OR ROTOR MAGNETS</h1>

In [ ]:
#For now my knowledge is not sufficient to understand how skewing will help reduce noise ripple, torque pulsation etc, so for now will be skipped

M<h1>STATOR SLOT AND WINDING DESIGN</h1>

In [54]:
FRACTIONAL = 1
INTEGRAL = 0
f_fill = 0.4                                # Slot fill factor [-]
a = 2                                       # Number of parallel paths [-]
cos_phi = cos_delta_s * 0.01 + cos_delta_s  # Corrected cos_delta for 1 %
J = 5 * 1e6                                 # Current density [A/mm^2], 1e6 to put it into [A/m^2]
if FRACTIONAL:
    if (2*p) % (a*b_q) != 0:
        raise ValueError("Not enough radial force available")
if INTEGRAL:
    if (2*p) % a != 0:
        raise ValueError("Not enough radial force available")

U = U_l_l / math.sqrt(3)                # RMS phase voltage in Y configuration [V]
print(f"RMS phase voltage in Y configuration U = {U:.2f} V")
E = 0.98 * U * cos_delta_s              # BACK-EMF voltage phasor [V], 0.98 is to include the stator resistance
print(f"Back-EMF voltage phasor E = {E:.2f} V")
Phi_m = (D * l * B_delta_m_1) / p       # Flux per pole pitch [Vs] or [Wb]
z = round(E / (2.22 * Phi_m * f_w * f))        # Number of conductors per phase connected in series [-]
z_slot = round((m * z * a) / Q_s)
print(f"Number of conductors per slot z_slot = {z_slot} ")
z = (z_slot * Q_s) / (m*a)
print(f"Number of conductors per phase connected in series z = {z:.2f} ")
Phi_m_corr = E / (2.22 * z * f_w * f)
print(f"Flux per pole pitch Phi_m = {Phi_m_corr:.6f} Vs")
B_delta_m_1_corr = (Phi_m_corr * p) / (D * l)
print(f"B_delta_m_1 = {B_delta_m_1_corr:.6f} T")
B_delta_m_corr = (math.pi / 4) * B_delta_m_1_corr / math.sin((alpha_m / 2) * (math.pi / 180))
print(f"B_delta_m = {B_delta_m_corr:.6f} T")
h_m_corr = (delta_marked * mi_r) / ((Br / B_delta_m) - 1)
print(f"h_m = {h_m_corr * 1e3:.3f} mm")
I = P2 / (3 * U * eta * cos_phi)
print(f"I = {I:.2f} A")
q_c = I / (a*J)
print(f"Required cross section of a conductor is = {q_c*1e6:.9f} mm^2")
S_Cu = z_slot * q_c
print(f"Cross sectional area of the copper is S_Cu = {S_Cu*1e6:.2f} mm^2")
S_slot = S_Cu / f_fill
print(f"Total are of the slot S_slot = {S_slot*1e6} mm^2 ")

RMS phase voltage in Y configuration U = 28.87 V
Back-EMF voltage phasor E = 26.03 V
Number of conductors per slot z_slot = 98 
Number of conductors per phase connected in series z = 294.00 
Flux per pole pitch Phi_m = 0.000844 Vs
B_delta_m_1 = 0.978440 T
B_delta_m = 0.847907 T
h_m = 2.918 mm
I = 14.19 A
Required cross section of a conductor is = 1.418587279 mm^2
Cross sectional area of the copper is S_Cu = 139.02 mm^2
Total are of the slot S_slot = 347.5538833214861 mm^2 
